# understudy - train the voice, then check what it learned

Runs on a free Kaggle T4. Nothing here needs an API key.

Before you start:
1. **Settings -> Accelerator -> GPU T4 x2**, and **Internet -> On** (both need a
   phone-verified account).
2. Add the dataset you uploaded with `train.jsonl`, `heldout.jsonl`,
   `stranger.jsonl` and `facts.json` in it.

This notebook renders no verdict. It writes the raw numbers to
`responses.json`; your laptop decides what they mean.

## 1. Install

Copied from Unsloth's own Kaggle notebook. Do not hand-edit the pins - this
stack breaks in silent ways when the versions drift apart. If this cell errors,
take the current one from
[unslothai/notebooks](https://github.com/unslothai/notebooks/tree/main/kaggle).

In [ ]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.5" transformers==5.5.0 torchcodec timm

## 2. Find your data

Kaggle mounts every added dataset under `/kaggle/input`. The search below means
you do not have to care what you named yours.

In [ ]:
import json, glob, os

def find(name):
    """Locate one of our four files wherever Kaggle mounted it."""
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(
            f"{name} is not in /kaggle/input. Add your dataset with the panel "
            f"on the right, then re-run this cell.")
    return hits[0]

def read_jsonl(name):
    with open(find(name), encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

train_rows    = read_jsonl("train.jsonl")
heldout_rows  = read_jsonl("heldout.jsonl")
stranger_rows = read_jsonl("stranger.jsonl")
with open(find("facts.json"), encoding="utf-8") as fh:
    facts = json.load(fh)

print(f"{len(train_rows)} training pairs")
print(f"{len(heldout_rows)} held out (yours), {len(stranger_rows)} (the control)")
print(f"{len(facts)} fact questions")
print("\nfirst training pair:")
for turn in train_rows[0]["messages"]:
    print(f"  {turn['role']:9} {turn['content'][:70]}")

## 3. Load the model

`Qwen3-4B-Instruct-2507` is ungated and Apache-2.0, so there is no token and no
licence click. In 4-bit it leaves room on a 16 GB T4 for a batch and the LoRA.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL = "unsloth/Qwen3-4B-Instruct-2507"
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

## 4. Attach the LoRA

The base weights stay frozen. These adapters are the only thing that trains -
a few tens of millions of numbers against four billion, which is why this fits
on a free GPU at all.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 5. Apply the chat template

Your JSONL is already `{"messages": [{"role", "content"}, ...]}`, which is what
`apply_chat_template` wants, so it goes straight in.

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-instruct")

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize = False,
                                                   add_generation_prompt = False)
                     for m in batch["messages"]]}

dataset = Dataset.from_list(train_rows).map(to_text, batched = True)
print(dataset[0]["text"][:600])

## 6. Train

Sixty steps is about five and a half minutes on a T4 x2 at this dataset size.

`train_on_responses_only` masks the other person's turns so the loss is only
ever computed on your replies - without it the model learns to imitate both
sides of the conversation equally.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)
trainer = train_on_responses_only(trainer)

### Check the masking before you spend twelve minutes on it

`train_on_responses_only` fails **silently** when it cannot find the template's
turn markers: every label comes back `-100`, the loss is zero, and training
completes having changed nothing. This cell makes that failure loud.

In [ ]:
sample = trainer.train_dataset[0]
labels = sample["labels"]
kept = [t for t in labels if t != -100]

assert kept, ("Every label is masked, so the loss would be zero and training "
              "would change nothing. The chat template and the masker disagree "
              "about this model's turn markers.")
print(f"{len(kept)} of {len(labels)} tokens are trained on "
      f"({len(kept)/len(labels):.0%})")
print("\nwhat the loss is computed on:")
print(tokenizer.decode([t for t in kept]))

In [ ]:
stats = trainer.train()
final_loss = stats.training_loss
print(f"\nfinal training loss {final_loss:.4f}")
assert final_loss > 0, "Zero loss means nothing trained."

## 7. Two models, one set of weights

The base arm is the same object with the adapters switched off. That matters
for a fair comparison: identical tokenizer, identical quantisation, identical
everything except the thing being measured.

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def reply_nll(messages):
    """Average loss per token on the final assistant turn, given its context."""
    full   = tokenizer.apply_chat_template(messages, tokenize = False,
                                           add_generation_prompt = False)
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize = False,
                                           add_generation_prompt = True)
    ids = tokenizer(full, return_tensors = "pt",
                    add_special_tokens = False).input_ids.to(model.device)
    cut = tokenizer(prompt, return_tensors = "pt",
                    add_special_tokens = False).input_ids.shape[1]
    if ids.shape[1] <= cut or ids.shape[1] > MAX_SEQ:
        return None
    labels = ids.clone()
    labels[:, :cut] = -100                      # score the reply, not the prompt
    logits = model(input_ids = ids).logits
    loss = F.cross_entropy(
        logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
        labels[:, 1:].reshape(-1), ignore_index = -100)
    return float(loss), int((labels[:, 1:] != -100).sum())

@torch.no_grad()
def say(messages, max_new_tokens = 80):
    """Greedy generation, so the same input always gives the same answer."""
    prompt = tokenizer.apply_chat_template(messages, tokenize = False,
                                           add_generation_prompt = True)
    batch = tokenizer(prompt, return_tensors = "pt",
                      add_special_tokens = False).to(model.device)
    out = model.generate(**batch, max_new_tokens = max_new_tokens,
                         do_sample = False,
                         pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id)
    return tokenizer.decode(out[0][batch.input_ids.shape[1]:],
                            skip_special_tokens = True).strip()

### Prove the two arms are actually different

If `disable_adapter()` is not doing what we think, both columns of every table
would be the same number and the whole comparison would be meaningless.

In [ ]:
probe = heldout_rows[0]["messages"]
tuned_check = reply_nll(probe)
with model.disable_adapter():
    base_check = reply_nll(probe)

print(f"base  {base_check[0]:.4f}")
print(f"tuned {tuned_check[0]:.4f}")
assert abs(base_check[0] - tuned_check[0]) > 1e-4, (
    "Both arms scored identically, so the adapter is not switching off. "
    "Every comparison below would be vacuous.")

## 8. Measure

Three things, in both arms: how surprised each model is by real held-out
replies of yours, the same for the other person in the chat, and what each one
says when asked about facts that are in the training data.

The cells below print `Both max_new_tokens and max_length seem to have been
set` once per generation. It is noise - `max_new_tokens` wins, which is what we
want - and there is one line of it per call, so expect a wall of them.

In [ ]:
from tqdm.auto import tqdm

def score_set(rows, prefix, limit = 40):
    out = []
    for i, row in enumerate(tqdm(rows[:limit], desc = prefix)):
        messages = row["messages"]
        tuned = reply_nll(messages)
        if tuned is None:
            continue
        with model.disable_adapter():
            base = reply_nll(messages)
        out.append({"pair_id": f"{prefix}{i:04d}", "tokens": tuned[1],
                    "nll_base": round(base[0], 4), "nll_tuned": round(tuned[0], 4)})
    return out

style_you      = score_set(heldout_rows,  "p")
style_stranger = score_set(stranger_rows, "c")
print(f"scored {len(style_you)} of yours and {len(style_stranger)} of theirs")

In [ ]:
generations = []
for i, row in enumerate(tqdm(heldout_rows[:8], desc = "generations")):
    messages = row["messages"]
    context, real = messages[:-1], messages[-1]["content"]
    tuned = say(context)
    with model.disable_adapter():
        base = say(context)
    generations.append({"pair_id": f"p{i:04d}",
                        "context": context[-1]["content"],
                        "real": real, "base": base, "tuned": tuned})

for g in generations[:3]:
    print(f"\nthey said  {g['context'][:90]}")
    print(f"you said   {g['real'][:90]}")
    print(f"base       {g['base'][:90]}")
    print(f"tuned      {g['tuned'][:90]}")

In [ ]:
fact_answers = []
for fact in tqdm(facts, desc = "facts"):
    ask = [{"role": "user", "content": fact["question"]}]
    tuned = say(ask, max_new_tokens = 48)
    with model.disable_adapter():
        base = say(ask, max_new_tokens = 48)
    fact_answers.append({"id": fact["id"], "question": fact["question"],
                         "base": base, "tuned": tuned})

for a in fact_answers[:5]:
    print(f"\nQ {a['question']}")
    print(f"  base  {a['base'][:80]}")
    print(f"  tuned {a['tuned'][:80]}")

## 9. Write `responses.json`

Numbers only. No pass, no fail, no verdict - those are decisions, and decisions
belong somewhere you can change your mind for free.

Download this file from the panel on the right, drop it in the project's
`data/` folder, and run `score` on your laptop.

In [ ]:
payload = {
    "schema": "understudy/responses/1",
    "run": {
        "base_model": MODEL,
        "steps": 60,
        "lora_r": 32,
        "trained_on": len(train_rows),
        "final_loss": round(final_loss, 4),
        "stranger": "the other person",
    },
    "style": {"you": style_you, "stranger": style_stranger},
    "generations": generations,
    "facts": fact_answers,
}

with open("/kaggle/working/responses.json", "w", encoding="utf-8") as fh:
    json.dump(payload, fh, ensure_ascii=False, indent=2)

print("wrote /kaggle/working/responses.json")
print("Download it, then run:")
print('  PYTHONPATH=. uv run python -m src.main score --responses data/responses.json')

## 10. Optional - keep the model

Everything above is done and the hour is spent. This last part is for if you
want the model on your own machine afterwards rather than losing it when the
Kaggle session ends.

It converts the merged weights to GGUF, which Ollama reads. Budget another
fifteen minutes here plus a 2.5 GB download.

In [ ]:
model.save_pretrained("understudy_lora")       # adapters only, a few hundred MB
tokenizer.save_pretrained("understudy_lora")
print("adapters saved")

In [ ]:
# Slow. Skip it unless you actually want the model locally.
model.save_pretrained_gguf("understudy_gguf", tokenizer,
                           quantization_method = "q4_k_m")
print("GGUF written to understudy_gguf/")

Two things about that output worth knowing before you go looking for the file.

The folder gets the suffix twice - it is `understudy_gguf_gguf/`, not
`understudy_gguf/`. And the `.gguf` inside is named after the **base model**,
not after your run, so it is `qwen3-4b-instruct-2507.Q4_K_M.gguf`. Nothing in
the filename tells you it has your voice in it; label it yourself when you
download it.

Unsloth also writes a working `Modelfile` next to it with the right chat
template already filled in, so do not hand-write one. Download both files into
the same folder, then:

```
ollama create understudy -f Modelfile
ollama run understudy
```

If you moved the `.gguf` or renamed it, open the `Modelfile` and fix its `FROM`
line to match.